# US30 vs DXY — Equity-vs-FX style (no JPMaQS API)

**Goal:** Replicate the *strategy idea* from *Alpha through equity-versus-FX trading* using only **US30** (equity) and **DXY** (dollar index) data. No DataQuery/JPMaQS API keys required.

**Inputs:** Your US30 and DXY datasets (CSV or yfinance). Output: vol-targeted equity-vs-FX spread, simple PnL, and correlation.

## 1. Setup

In [ ]:
!pip install -q pandas numpy matplotlib

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os

## 2. Load US30 and DXY data

**Option A — Your CSV files:** Set paths below. Each CSV should have a date column (e.g. `Date`, `Timestamp`, `Time`) and `Close` (and optionally `Open`, `High`, `Low`, `Volume`).

**Option B — yfinance (no files):** If paths are empty, we download ^DJI and DX-Y.NYB (DXY) via yfinance.

In [ ]:
# Paths to your US30 and DXY CSVs (leave empty to use yfinance)
US30_CSV = os.environ.get("US30_CSV_PATH", "")  # e.g. "path/to/us30_daily.csv"
DXY_CSV  = os.environ.get("DXY_CSV_PATH",  "")  # e.g. "path/to/dxy_daily.csv"

In [ ]:
def _ensure_datetime_index(df, date_col=None):
    for c in (date_col,) if date_col else ["Timestamp", "Date", "Time", "datetime"]:
        if c and c in df.columns:
            df = df.copy()
            df[c] = pd.to_datetime(df[c])
            df = df.set_index(c).sort_index()
            return df
    if isinstance(df.index, pd.DatetimeIndex):
        return df.sort_index()
    raise ValueError("Need a date column: Timestamp, Date, or Time")

def load_series(csv_path, close_col="Close"):
    """Load one CSV and return a daily close series with DatetimeIndex."""
    df = pd.read_csv(csv_path)
    df.columns = [c.strip() for c in df.columns]
    if close_col not in df.columns:
        cand = [c for c in df.columns if "close" in c.lower() or c == "close"]
        close_col = cand[0] if cand else df.columns[-1]
    df = _ensure_datetime_index(df)
    if df.index.duplicated().any():
        df = df[~df.index.duplicated(keep="first")]
    return df[close_col].astype(float)

def load_us30_dxy_from_csv(us30_path, dxy_path):
    us30 = load_series(us30_path)
    dxy  = load_series(dxy_path)
    return us30, dxy

def load_us30_dxy_yfinance(start="2010-01-01", end=None):
    try:
        import yfinance as yf
    except ImportError:
        raise ImportError("Install yfinance: pip install yfinance")
    us30 = yf.download("^DJI", start=start, end=end, progress=False)["Close"]
    dxy  = yf.download("DX-Y.NYB", start=start, end=end, progress=False)["Close"]
    return us30.squeeze(), dxy.squeeze()

In [ ]:
if US30_CSV and DXY_CSV and os.path.isfile(US30_CSV) and os.path.isfile(DXY_CSV):
    us30_close, dxy_close = load_us30_dxy_from_csv(US30_CSV, DXY_CSV)
    print("Loaded from CSV:", US30_CSV, DXY_CSV)
else:
    us30_close, dxy_close = load_us30_dxy_yfinance(start="2010-01-01")
    print("Loaded via yfinance (^DJI, DX-Y.NYB)")

# Align to common dates
common = us30_close.index.intersection(dxy_close.index)
us30_close = us30_close.reindex(common).ffill().bfill()
dxy_close  = dxy_close.reindex(common).ffill().bfill()
print("US30 shape:", us30_close.shape, "DXY shape:", dxy_close.shape)

## 3. Volatility-targeted returns and equity-vs-FX spread

Same idea as the paper: scale each leg to ~10% annualized vol, then take **equity minus FX** (long US30, short dollar).

In [ ]:
TARGET_VOL = 0.10   # 10% annualized
ROLLING_W = 21      # ~1 month for daily data

def vol_target_returns(close_series, target_vol=TARGET_VOL, window=ROLLING_W):
    ret = close_series.pct_change().dropna()
    vol = ret.rolling(window).std() * np.sqrt(252)  # annualized
    vol = vol.replace(0, np.nan).ffill().bfill()
    scale = target_vol / vol
    return (ret * scale).dropna()

eq_ret = vol_target_returns(us30_close)
fx_ret = vol_target_returns(dxy_close)

# Equity vs FX: long equity, short dollar (so we subtract DXY return)
eq_vs_fx = eq_ret.sub(fx_ret.reindex(eq_ret.index).ffill().bfill()).dropna()
print("Equity-vs-FX return series length:", len(eq_vs_fx))

## 4. Naive PnL and correlation

Constant long equity-vs-FX (no macro signals).

In [ ]:
pnl = eq_vs_fx.cumsum()
ann_vol = eq_vs_fx.std() * np.sqrt(252)
ann_ret = eq_vs_fx.mean() * 252
sharpe = ann_ret / ann_vol if ann_vol else 0
print(f"Ann. return (approx): {ann_ret:.4f}")
print(f"Ann. vol (approx):    {ann_vol:.4f}")
print(f"Sharpe (approx):      {sharpe:.2f}")
print(f"Corr(US30 ret, DXY ret): {eq_ret.corr(fx_ret.reindex(eq_ret.index)):.3f}")

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(10, 6), sharex=True)
eq_vs_fx.rolling(21).mean().plot(ax=axes[0], title="Equity-vs-FX return (rolling 21d mean)")
pnl.plot(ax=axes[1], title="Cumulative PnL (long US30 vs short DXY, vol-targeted)")
plt.tight_layout()
plt.show()

## 5. What you need (summary)

- **Data:** US30 and DXY in the same frequency (e.g. daily). Either:
  - Two CSVs with a date column and `Close`, or
  - `pip install yfinance` and use this notebook without CSVs.
- **Env:** `pandas`, `numpy`, `matplotlib`. Optional: `yfinance` if not using CSVs.
- **No API keys** required. Macro signals from the original research are not available without JPMaQS; you can later add your own signals (e.g. momentum, mean reversion) on top of this spread.